In [ ]:
import subprocess

def run(command):
    """Run a shell command, show its output live, and fail the notebook if the command fails."""
    print("$", command, flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"exit code {process.returncode}: {command}")


In [ ]:
run("nvidia-smi")
run("pip install -q transformers==4.57.1 peft==0.17.1 accelerate")

In [ ]:
%%writefile convert_mapqa.py
#!/usr/bin/env python3
"""Convert MapQA-S into derenderer training files: one map image + the JSON the model should output.

Continuous legend (smooth color bar):
    {"title":"The Percentage of ceremonies in the USA","legend_title":"Value","legend_type":"continuous",
     "data_type":"percentage","values":{"AK":37.9,"AL":39.8,...,"WY":47.4}}

Discrete legend (ranges):
    {"title":"The Number of shots in the USA","legend_title":null,"legend_type":"discrete","data_type":"number",
     "ranges":["7548477-7658039","7668882-7761001","7811458-7859166","7920264-8002936"],
     "values":{"AK":0,"AL":0,...,"WY":1}}

Rules:
- values are keyed by state code, sorted, all 50 states every time
- discrete: each state gets its range number (0 = lowest range); "ranges" lists the legend ranges lowest
  first, whatever order the image shows them in; states with no data get null (the image shows "N/A")
- continuous: numbers are rounded so the color bar has about 100 steps, since the image can't show more
- legend_title is the text above the legend: "Value" on continuous maps, nothing on discrete maps
- data_type is "percentage" or "number": every MapQA title says which, and percentage ranges carry "%"
- train/dev/test follow MapQA's own question files; the few maps that have no questions are skipped

Usage:
    python mapqa/convert_mapqa.py --mapqa ~/Downloads/MapQA_S.zip --out data/mapqa_derender
    python mapqa/convert_mapqa.py --mapqa /kaggle/input/<your-dataset> --out /kaggle/working/mapqa_derender

--mapqa can be the zip or any folder that contains MapQA_S/.

Output: <out>/train.jsonl, dev.jsonl, test.jsonl with one line per map:
    {"id": "map_10", "image": "images/map_10.png", "target": "<the JSON above, as a string>",
     "legend_type": "discrete", "data_type": "number", "true_values": {"AK": 7603168, ...}}
"image" is relative to the MapQA_S folder. true_values are the exact numbers, for evaluation only.
"""

import argparse
import json
import math
import re
import statistics
import zipfile
from pathlib import Path

SPLITS = ("train", "dev", "test")
CONTINUOUS_LEGEND_TITLE = "Value"  # checked on several MapQA-S images; discrete maps show no legend title


class MapQASource:
    """Reads MapQA_S files from the downloaded zip or from an extracted folder."""

    def __init__(self, path):
        path = Path(path).expanduser()
        if path.is_file() and zipfile.is_zipfile(path):
            self.zip = zipfile.ZipFile(path)
            names = [n for n in self.zip.namelist() if not n.startswith("__MACOSX")]
            first_meta = next((n for n in names if re.search(r"(^|/)metadata/map_\d+\.json$", n)), None)
            if first_meta is None:
                raise FileNotFoundError(f"no metadata/map_*.json inside {path}")
            self.prefix = first_meta[: first_meta.index("metadata/")]
            self.names = {n[len(self.prefix):] for n in names if n.startswith(self.prefix)}
        else:
            self.zip = None
            patterns = ["*/" * depth + "metadata" for depth in range(5)]
            root = next((p.parent for pattern in patterns for p in path.glob(pattern)
                         if p.is_dir() and "__MACOSX" not in p.parts), None)
            if root is None:
                raise FileNotFoundError(f"no metadata/ folder found under {path}")
            self.root = root
        self.location = path

    def read(self, rel):
        if self.zip:
            return self.zip.read(self.prefix + rel)
        return (self.root / rel).read_bytes()

    def exists(self, rel):
        return rel in self.names if self.zip else (self.root / rel).exists()

    def metadata_ids(self):
        if self.zip:
            found = (re.fullmatch(r"metadata/map_(\d+)\.json", n) for n in self.names)
        else:
            found = (re.fullmatch(r"map_(\d+)\.json", p.name) for p in (self.root / "metadata").iterdir())
        return sorted(int(m.group(1)) for m in found if m)


def split_of_each_map(source):
    """map id -> split, from the map ids referenced in MapQA's question files."""
    split_of = {}
    for split in SPLITS:
        raw = source.read(f"questions/{split}-QA.json")
        for map_id in re.findall(rb'"map_id"\s*:\s*"map_(\d+)\.png"', raw):
            split_of[int(map_id)] = split
    return split_of


def is_missing(value):
    return value is None or (isinstance(value, float) and math.isnan(value))


def round_for_color_bar(values):
    """Round so the color bar spans about 100 steps: one decimal for a span of ~10, hundreds for ~10,000."""
    present = [v for v in values if v is not None]
    span = max(present) - min(present) if present else 0
    if span <= 0:
        return values
    digits = round(-math.log10(span / 100))
    if digits > 0:
        return [None if v is None else round(v, digits) for v in values]  # never adds digits to the data
    return [None if v is None else int(round(v, digits)) for v in values]


def convert_map(meta):
    """Return (target dict, true values) for one MapQA metadata entry."""
    map_id = meta["map_id"]
    data = sorted(meta["data"], key=lambda d: d["region"])
    if len(data) != 50:
        raise ValueError(f"map {map_id}: expected 50 states, got {len(data)}")
    codes = [d["region"] for d in data]
    true_values = {c: None if is_missing(d["value"]) else d["value"] for c, d in zip(codes, data)}
    title = meta["title"]["text"]
    data_type = meta["data_type"].lower()
    if data_type not in ("percentage", "number"):
        raise ValueError(f"map {map_id}: unknown data_type {meta['data_type']!r}")

    if meta["legend_type"] == "Continuous":
        values = round_for_color_bar([true_values[c] for c in codes])
        target = {"title": title, "legend_title": CONTINUOUS_LEGEND_TITLE, "legend_type": "continuous",
                  "data_type": data_type, "values": dict(zip(codes, values))}
    elif meta["legend_type"] == "Discrete":
        ranges = [c for c in meta["class_descriptions"] if c != "N/A"]
        classes = []
        for d in data:
            if d["class"] < 0:
                if d["description"] != "N/A":
                    raise ValueError(f"map {map_id}: {d['region']} has class {d['class']} but {d['description']!r}")
                classes.append(None)
            else:
                if ranges[d["class"]] != d["description"]:
                    raise ValueError(f"map {map_id}: {d['region']} class {d['class']} != {d['description']!r}")
                classes.append(d["class"])
        target = {"title": title, "legend_title": None, "legend_type": "discrete", "data_type": data_type,
                  "ranges": ranges, "values": dict(zip(codes, classes))}
    else:
        raise ValueError(f"map {map_id}: unknown legend_type {meta['legend_type']!r}")
    return target, true_values


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--mapqa", required=True, help="MapQA_S.zip, or a folder containing MapQA_S/")
    parser.add_argument("--out", type=Path, default=Path("data/mapqa_derender"))
    parser.add_argument("--limit", type=int, help="convert only the first N maps (quick test)")
    args = parser.parse_args()

    source = MapQASource(args.mapqa)
    split_of = split_of_each_map(source)
    ids = source.metadata_ids()[: args.limit]
    args.out.mkdir(parents=True, exist_ok=True)

    files = {split: open(args.out / f"{split}.jsonl", "w", encoding="utf-8") for split in SPLITS}
    counts = {split: {"continuous": 0, "discrete": 0} for split in SPLITS}
    no_split, no_image, lengths = [], [], []
    try:
        for map_id in ids:
            split = split_of.get(map_id)
            if split is None:
                no_split.append(map_id)
                continue
            image = f"images/map_{map_id}.png"
            if not source.exists(image):
                no_image.append(map_id)
                continue
            meta = json.loads(source.read(f"metadata/map_{map_id}.json"))
            target, true_values = convert_map(meta)
            target_text = json.dumps(target, ensure_ascii=False, separators=(",", ":"))
            record = {
                "id": f"map_{map_id}",
                "image": image,
                "target": target_text,
                "legend_type": target["legend_type"],
                "data_type": target["data_type"],
                "true_values": true_values,
            }
            files[split].write(json.dumps(record, ensure_ascii=False) + "\n")
            counts[split][target["legend_type"]] += 1
            lengths.append(len(target_text))
    finally:
        for f in files.values():
            f.close()

    print(f"Read {len(ids)} maps from {source.location}")
    for split in SPLITS:
        c = counts[split]
        print(f"  {split:5} {c['continuous'] + c['discrete']:6} maps "
              f"({c['discrete']} discrete, {c['continuous']} continuous) -> {args.out / (split + '.jsonl')}")
    if no_split:
        print(f"  skipped {len(no_split)} map(s) that no question file uses: {no_split[:10]}")
    if no_image:
        print(f"  skipped {len(no_image)} map(s) with no image: {no_image[:10]}")
    if lengths:
        print(f"  target length in characters: min {min(lengths)}, median {int(statistics.median(lengths))}, "
              f"max {max(lengths)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile common.py
"""Shared pieces of the derenderer training scripts: loading maps, the training loop, and scoring.

Training data is MapQA (from convert_mapqa.py) plus, with --synth, synthetic maps from
synth/generate_maps.py. Both use the same target JSON.

Scoring (per legend type; MapQA and synthetic test maps are scored separately):
    parsed                 output is a JSON object
    exact_match            output equals the target exactly
    title_correct          title matches (ignoring case and spacing)
    legend_title_correct   legend title matches (ignoring case and spacing; both null also counts)
    legend_type_correct    "continuous" / "discrete" is right
    data_type_correct      "percentage" / "number" is right
    coverage               share of the 50 states present in "values"
    extra_keys             keys in "values" that are not states of this map
  discrete maps:
    ranges_correct         the legend ranges match exactly
    state_accuracy         share of states with the right range index (null for no data)
  continuous maps (errors are relative to the true value range, so 0.05 = 5% of the range):
    nmae, nrmse            mean / root-mean-square error; a missing or non-numeric state counts as 1,
                           and no single error counts more than 1
    within_5pct            share of states within 5% of the range
Maps whose output doesn't parse count as 0 for accuracies and 1 for errors.
"""

import contextlib
import json
import math
import random
import time
from pathlib import Path

import torch
from PIL import Image

def add_common_args(parser):
    parser.add_argument("--data", type=Path, required=True, help="folder with MapQA train.jsonl and test.jsonl")
    parser.add_argument("--mapqa", type=Path, required=True, help="folder containing MapQA_S/images (searched a few levels)")
    parser.add_argument("--synth", type=Path, help="folder containing the synthetic maps (train.jsonl next to "
                                                   "images/synth_*, searched a few levels); adds them to training "
                                                   "and scoring")
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--train-maps", type=int, help="train on a random subset of this many MapQA maps")
    parser.add_argument("--synth-train-maps", type=int, help="train on a random subset of this many synthetic maps")
    parser.add_argument("--eval-maps", type=int, default=400, help="MapQA test maps to score (same maps for every model)")
    parser.add_argument("--synth-eval-maps", type=int, default=200, help="synthetic test maps to score")
    parser.add_argument("--epochs", type=int, default=2)
    parser.add_argument("--hours", type=float, default=8.0,
                        help="training time budget; the learning-rate schedule is shortened to fit it")
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--grad-accum", type=int, default=4)
    parser.add_argument("--lr", type=float, default=5e-5)
    parser.add_argument("--weight-decay", type=float, default=0.01)
    parser.add_argument("--eval-batch-size", type=int, default=8)
    parser.add_argument("--max-new-tokens", type=int, default=640, help="longest MapQA target is about 530 tokens")
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--log-every", type=int, default=20)
    parser.add_argument("--skip-train", action="store_true", help="only score the model as loaded")
    parser.add_argument("--seed", type=int, default=0)


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)


def find_images_root(path, pattern):
    """The folder whose images/ holds files matching pattern, searched a few levels under path.

    Finding datasets by their files rather than folder names works however Kaggle mounts or unpacks them.
    """
    for depth in range(5):
        hits = Path(path).glob("*/" * depth + "images/" + pattern)
        hit = next((p for p in hits if "__MACOSX" not in p.parts), None)
        if hit:
            return hit.parent.parent
    raise FileNotFoundError(f"no images/{pattern} under {path}")


def find_mapqa_root(path):
    return find_images_root(path, "map_*.png")


def read_split(data_dir, split, image_root):
    """Records of one split, each with image_path set (a record's "image" is relative to image_root)."""
    records = []
    with open(Path(data_dir) / f"{split}.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            record["image_path"] = str(Path(image_root) / record["image"])
            records.append(record)
    return records


def sample_records(records, n, seed):
    if n is None or n >= len(records):
        return list(records)
    return random.Random(seed).sample(records, n)


def training_records(args, mapqa_root):
    records = sample_records(read_split(args.data, "train", mapqa_root), args.train_maps, args.seed)
    synthetic = []
    if args.synth:
        folder = find_images_root(args.synth, "synth_*")
        synthetic = sample_records(read_split(folder, "train", folder), args.synth_train_maps, args.seed)
    print(f"training maps: {len(records)} MapQA + {len(synthetic)} synthetic", flush=True)
    return records + synthetic


def evaluation_sets(args, mapqa_root):
    """(name, records) pairs: MapQA test maps, plus synthetic test maps when --synth is given."""
    sets = [("mapqa", sample_records(read_split(args.data, "test", mapqa_root), args.eval_maps, seed=0))]
    if args.synth:
        folder = find_images_root(args.synth, "synth_*")
        sets.append(("synth", sample_records(read_split(folder, "test", folder), args.synth_eval_maps, seed=0)))
    return sets


class MapDataset(torch.utils.data.Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        record = self.records[i]
        return Image.open(record["image_path"]).convert("RGB"), record["target"]


def as_list(batch):
    return batch


def autocast(device, amp_dtype):
    return torch.autocast(device.type, dtype=amp_dtype) if amp_dtype else contextlib.nullcontext()


def train_loop(model, loader, args, forward, out_dir, amp_dtype=None):
    """AdamW with warmup + cosine decay. Stops after --epochs or --hours, whichever comes first.

    forward(batch) returns the loss. After 20 updates the schedule is shortened, if needed, so the
    learning rate still decays to zero within the time budget.
    """
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=args.weight_decay)
    schedule = {"total": max(1, args.epochs * math.ceil(len(loader) / args.grad_accum))}

    def lr_factor(update):
        warmup = max(1, round(0.03 * schedule["total"]))
        if update < warmup:
            return (update + 1) / warmup
        progress = min(1.0, (update - warmup) / max(1, schedule["total"] - warmup))
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16)
    device = next(model.parameters()).device
    start = time.time()
    deadline = start + args.hours * 3600
    update, update_loss, window = 0, 0.0, []
    model.train()
    with open(out_dir / "train_log.jsonl", "a") as log:
        for epoch in range(args.epochs):
            for i, batch in enumerate(loader, 1):
                with autocast(device, amp_dtype):
                    loss = forward(batch)
                scaler.scale(loss / args.grad_accum).backward()
                update_loss += loss.item() / args.grad_accum
                if i % args.grad_accum and i != len(loader):
                    continue
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                update += 1
                window.append(update_loss)
                update_loss = 0.0
                if update == 20:
                    seconds_per_update = (time.time() - start) / update
                    fits = update + int((deadline - time.time()) / seconds_per_update)
                    schedule["total"] = max(update + 1, min(schedule["total"], fits))
                if update == 1 or update % args.log_every == 0:
                    entry = {"epoch": epoch, "update": update, "of": schedule["total"],
                             "loss": round(sum(window) / len(window), 4), "lr": scheduler.get_last_lr()[0],
                             "minutes": round((time.time() - start) / 60, 1)}
                    window.clear()
                    print(json.dumps(entry), flush=True)
                    log.write(json.dumps(entry) + "\n")
                    log.flush()
                if update >= schedule["total"] or time.time() > deadline:
                    print(f"stopping after {update} updates ({(time.time() - start) / 3600:.2f} h)", flush=True)
                    return update
    return update


# ---------------------------------------------------------------- scoring

def parse_prediction(text):
    """The first JSON object in the model output, or None."""
    start = text.find("{")
    if start < 0:
        return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(text[start:])
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None


def _same_text(a, b):
    if isinstance(a, str) and isinstance(b, str):
        return " ".join(a.split()).lower() == " ".join(b.split()).lower()
    return a is None and b is None


def _is_number(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v)


def score_map(record, prediction_text):
    target = json.loads(record["target"])
    pred = parse_prediction(prediction_text)
    score = {"id": record["id"], "legend_type": target["legend_type"], "parsed": pred is not None}
    if pred is None:
        return score
    values = pred.get("values") if isinstance(pred.get("values"), dict) else {}
    codes = list(target["values"])
    score["exact_match"] = pred == target
    score["title_correct"] = _same_text(pred.get("title"), target["title"])
    score["legend_title_correct"] = _same_text(pred.get("legend_title"), target["legend_title"])
    score["legend_type_correct"] = pred.get("legend_type") == target["legend_type"]
    score["data_type_correct"] = pred.get("data_type") == target["data_type"]
    score["coverage"] = sum(c in values for c in codes) / len(codes)
    score["extra_keys"] = sum(k not in target["values"] for k in values)
    if target["legend_type"] == "discrete":
        score["ranges_correct"] = pred.get("ranges") == target["ranges"]
        score["state_accuracy"] = sum(c in values and values[c] == target["values"][c] for c in codes) / len(codes)
    else:
        truth = {c: v for c, v in record["true_values"].items() if v is not None}
        span = (max(truth.values()) - min(truth.values())) or 1.0
        errors = [min(1.0, abs(values[c] - v) / span) if _is_number(values.get(c)) else 1.0 for c, v in truth.items()]
        score["nmae"] = sum(errors) / len(errors)
        score["nrmse"] = math.sqrt(sum(e * e for e in errors) / len(errors))
        score["within_5pct"] = sum(e <= 0.05 for e in errors) / len(errors)
    return score


def summarize(scores):
    summary = {}
    for kind in ("discrete", "continuous"):
        group = [s for s in scores if s["legend_type"] == kind]
        if not group:
            continue

        def mean(key, if_unparsed):
            return round(sum(float(s.get(key, if_unparsed)) for s in group) / len(group), 4)

        result = {"maps": len(group)}
        for key in ("parsed", "exact_match", "title_correct", "legend_title_correct", "legend_type_correct",
                    "data_type_correct", "coverage"):
            result[key] = mean(key, 0)
        result["extra_keys"] = mean("extra_keys", 0)
        if kind == "discrete":
            result.update(ranges_correct=mean("ranges_correct", 0), state_accuracy=mean("state_accuracy", 0))
        else:
            result.update(nmae=mean("nmae", 1), nrmse=mean("nrmse", 1), within_5pct=mean("within_5pct", 0))
        summary[kind] = result
    return summary


def evaluate_and_save(records, predictions, out_dir, name):
    """Score one test set; writes predictions_<name>.jsonl and metrics_<name>.json."""
    scores = [score_map(r, p) for r, p in zip(records, predictions)]
    with open(out_dir / f"predictions_{name}.jsonl", "w", encoding="utf-8") as f:
        for record, prediction, score in zip(records, predictions, scores):
            f.write(json.dumps({"id": record["id"], "prediction": prediction, "target": record["target"],
                                "score": score}, ensure_ascii=False) + "\n")
    summary = summarize(scores)
    (out_dir / f"metrics_{name}.json").write_text(json.dumps(summary, indent=2))
    print(f"{name} test maps:\n{json.dumps(summary, indent=2)}", flush=True)
    return summary


In [ ]:
%%writefile train_deplot.py
#!/usr/bin/env python3
"""Fine-tune DePlot (google/deplot) to turn a choropleth map into the target JSON, then score it on test maps.

    python train_deplot.py --data /kaggle/working/data --mapqa /kaggle/input --out /kaggle/working/deplot
    python train_deplot.py ... --synth /kaggle/input     # also train and score on synthetic maps

Writes <out>/model (fine-tuned model), train_log.jsonl, and predictions_<set>.jsonl + metrics_<set>.json for
each test set (mapqa, plus synth when --synth is given).
"""

import argparse
import json
import time

import torch
from torch.utils.data import DataLoader
from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

from common import (MapDataset, add_common_args, as_list, autocast, evaluate_and_save, evaluation_sets,
                    find_mapqa_root, set_seed, train_loop, training_records)

PROMPT = "Generate underlying data table of the figure below:"  # the prompt DePlot was trained with


class TrainCollate:
    def __init__(self, processor, max_patches, max_target_tokens):
        self.processor = processor
        self.max_patches = max_patches
        self.max_target_tokens = max_target_tokens

    def __call__(self, batch):
        images, targets = zip(*batch)
        enc = self.processor(images=list(images), text=[PROMPT] * len(images), max_patches=self.max_patches,
                             return_tensors="pt")
        tokenizer = self.processor.tokenizer
        labels = tokenizer(list(targets), max_length=self.max_target_tokens, truncation=True, padding="longest",
                           return_tensors="pt").input_ids
        labels[labels == tokenizer.pad_token_id] = -100
        return enc["flattened_patches"], enc["attention_mask"], labels


def predict(model, processor, records, args, device, amp_dtype):
    loader = DataLoader(MapDataset(records), batch_size=args.eval_batch_size, num_workers=args.workers,
                        collate_fn=as_list)
    predictions = []
    began = time.time()
    with torch.no_grad():
        for batch in loader:
            images = [image for image, _ in batch]
            enc = processor(images=images, text=[PROMPT] * len(images), max_patches=args.max_patches,
                            return_tensors="pt").to(device)
            with autocast(device, amp_dtype):
                ids = model.generate(**enc, max_new_tokens=args.max_new_tokens, do_sample=False, num_beams=1)
            predictions += processor.batch_decode(ids, skip_special_tokens=True)
            print(f"predicted {len(predictions)}/{len(records)} ({(time.time() - began) / 60:.1f} min)", flush=True)
    return predictions


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    add_common_args(parser)
    parser.add_argument("--model", default="google/deplot")
    parser.add_argument("--max-patches", type=int, default=2048, help="a 700x500 map fits in 2048 patches at full size")
    parser.add_argument("--max-target-tokens", type=int, default=640, help="longest MapQA target is about 530 tokens")
    parser.add_argument("--fp16", action="store_true", help="mixed precision (off by default: T5-style models can overflow)")
    # Pix2Struct has no gradient checkpointing and 2 maps per batch ran out of memory on a 16 GB T4;
    # the effective batch is still 16
    parser.set_defaults(epochs=3, lr=5e-5, batch_size=1, grad_accum=16)
    args = parser.parse_args()
    set_seed(args.seed)
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / "args.json").write_text(json.dumps(vars(args), default=str, indent=1))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp_dtype = torch.float16 if args.fp16 and device.type == "cuda" else None
    root = find_mapqa_root(args.mapqa)
    processor = Pix2StructProcessor.from_pretrained(args.model)
    model = Pix2StructForConditionalGeneration.from_pretrained(args.model).to(device)

    if not args.skip_train:
        train = training_records(args, root)
        loader = DataLoader(MapDataset(train), batch_size=args.batch_size, shuffle=True, num_workers=args.workers,
                            collate_fn=TrainCollate(processor, args.max_patches, args.max_target_tokens))

        def forward(batch):
            patches, mask, labels = (t.to(device) for t in batch)
            return model(flattened_patches=patches, attention_mask=mask, labels=labels).loss

        if model.supports_gradient_checkpointing:
            model.gradient_checkpointing_enable()
        print(f"{len(loader)} batches per epoch", flush=True)
        train_loop(model, loader, args, forward, args.out, amp_dtype)
        model.save_pretrained(args.out / "model")
        processor.save_pretrained(args.out / "model")

    model.eval()
    for name, records in evaluation_sets(args, root):
        print(f"scoring {len(records)} {name} test maps", flush=True)
        evaluate_and_save(records, predict(model, processor, records, args, device, amp_dtype), args.out, name)


if __name__ == "__main__":
    main()


In [ ]:
run("python convert_mapqa.py --mapqa /kaggle/input --out /kaggle/working/data")

In [ ]:
run('python train_deplot.py --data /kaggle/working/data --mapqa /kaggle/input --out /kaggle/working/deplot --synth /kaggle/input --hours 1 --epochs 1 --eval-maps 24 --synth-eval-maps 8 --eval-batch-size 16 --max-new-tokens 560 --log-every 20')